In [10]:
## We gonna move forward to understand the data.

In [ ]:
pip install pandas

In [3]:
import pandas as pd

data = pd.read_json('../results/process_data.json')

In [ ]:
data

In [5]:
data.head()

,timestamp,temperature,pressure,velocity,humidity,qualityScore
0,2023-10-01 08:00:00,85.3,102.4,45.1,65.2,87.3
1,2023-10-01 09:00:00,86.1,103.1,46.0,64.8,88.1
2,2023-10-01 10:00:00,87.2,102.8,45.5,64.5,89.0
3,2023-10-01 11:00:00,88.5,102.9,45.2,64.0,89.5
4,2023-10-01 12:00:00,89.1,103.2,44.8,63.5,89.8


In [6]:
data.describe()

,timestamp,temperature,pressure,velocity,humidity,qualityScore
count,30,30.00000,30.000000,30.000000,30.000000,30.000000
mean,2023-10-02 12:30:00,88.87000,103.420000,44.490000,63.170000,89.593333
min,2023-10-01 08:00:00,85.20000,102.300000,42.700000,60.800000,87.200000
25%,2023-10-01 15:15:00,87.25000,102.925000,43.725000,62.025000,89.025000
50%,2023-10-02 12:30:00,89.40000,103.350000,44.650000,63.200000,89.950000
75%,2023-10-03 09:45:00,90.57500,103.975000,45.200000,64.450000,90.475000
max,2023-10-03 17:00:00,91.10000,104.400000,46.100000,65.400000,90.800000
std,NaN,1.95821,0.625548,0.998395,1.401268,1.096997


In [9]:
data.corr()

,timestamp,temperature,pressure,velocity,humidity,qualityScore
timestamp,1.000000,0.119194,0.086200,-0.111362,-0.111962,0.098309
temperature,0.119194,1.000000,0.906100,-0.860699,-0.955912,0.989368
pressure,0.086200,0.906100,1.000000,-0.881415,-0.967418,0.875053
velocity,-0.111362,-0.860699,-0.881415,1.000000,0.948226,-0.796616
humidity,-0.111962,-0.955912,-0.967418,0.948226,1.000000,-0.922554
qualityScore,0.098309,0.989368,0.875053,-0.796616,-0.922554,1.000000


* training data and test data

In [15]:
features = ['temperature', 'pressure', 'velocity', 'humidity']
X = data[features]
y = data['qualityScore']

In [17]:
X.head()

,temperature,pressure,velocity,humidity
0,85.3,102.4,45.1,65.2
1,86.1,103.1,46.0,64.8
2,87.2,102.8,45.5,64.5
3,88.5,102.9,45.2,64.0
4,89.1,103.2,44.8,63.5


In [19]:
y.head()

0    87.3
1    88.1
2    89.0
3    89.5
4    89.8
Name: qualityScore, dtype: float64

## Using Scikit-learn:
* we gonna use Scalers and Regressor for prediction.

In [13]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 22.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 38.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
X_train.head()

,temperature,pressure,velocity,humidity
28,90.8,104.1,43.3,61.6
24,89.0,103.1,44.9,63.6
12,87.4,102.9,45.6,64.3
0,85.3,102.4,45.1,65.2
4,89.1,103.2,44.8,63.5


In [23]:
X_test.head()

,temperature,pressure,velocity,humidity
27,90.5,103.9,43.8,62.1
15,89.8,103.6,44.4,62.8
23,88.4,102.8,45.3,64.1
17,90.7,104.1,43.6,61.8
8,90.9,104.2,43.2,61.5


In [24]:
y_train.head()

28    90.5
24    89.7
12    89.1
0     87.3
4     89.8
Name: qualityScore, dtype: float64

In [25]:
y_test.head()

27    90.4
15    90.2
23    89.4
17    90.6
8     90.6
Name: qualityScore, dtype: float64

In [26]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('regressor', RandomForestRegressor(random_state=42))])

## Getting the hyper-parameters:

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__n_estimators': [50, 100, 200], # numb of trees to select
    'regressor__max_depth': [None, 10, 20], # deep level
    'regressor__min_samples_split': [2, 5, 10] # min samples to split
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2') 
#cvÑ divides in n parts, to validate the robust of the possible combinations.
#scopring: coeficiente de determinación, I dont remember well this
    # que también se ajustan los valores predichos, al rango de valores suministrados, que tan real se ve con respecto a los datos de entrenamiento.

grid_search.fit(X_train, y_train)

print("the best hyper-parameters:", grid_search.best_params_)
print("the best score:", grid_search.best_score_)

the best hyper-parameters: {'regressor__max_depth': None, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 50}
the best score: 0.9822828266681105


## Validate the model and prediction:

In [29]:
from sklearn.metrics import r2_score, mean_squared_error

y_pred = grid_search.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("R² in test group:", r2)
print("error on test data:", mse)

R² in test group: 0.9811320970042791
error on test data: 0.003674000000000051


## Integrate with the optimization process:

In [31]:
pip install scikit-optimize


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
import numpy as np
from skopt import gp_minimize

def objective(params):
    temperature, pressure, velocity, humidity = params
    X_new = pd.DataFrame([[temperature, pressure, velocity, humidity]], columns=features)
    quality_pred = grid_search.predict(X_new)[0]
    return -quality_pred

space = [
    (80, 100),    # temperature
    (100, 110),   # pressure
    (40, 50),     # velocity
    (60, 70)      # humidity
]

res = gp_minimize(objective, space, n_calls=50, random_state=42)
print("the best parameters:", res.x)
print("best quality predited:", -res.fun)

the best parameters: [np.int64(93), np.int64(110), np.int64(40), np.int64(60)]
best quality predited: 90.73000000000002
